# Benchmarks - Model classes for Magma and MagmaClust

**Main considerations when implementing model classes**

* We want to provide simple abstractions for end users
* We want to provide flexibility for advanced users
* We want to provide smart defaults for most use cases
* We want to preserve performance and scalability
* We want to enhance the modularity of the codebase
* We want to adhere to class-based design principles

Throughout the MagmaClustPy framework:
* If it contains a "state" (e.g. parameters, hyperparameters, etc.), it should be a class
* If it has multiple variants that share the same interface (likelihoods, etc.), it should be a class heriting from an abstract base class
* If it is a mathematical "pure" function (e.g. likelihood, etc.), it should be a function

---
## Setup

In [1]:
# Jax configuration
USE_JIT = False
USE_X64 = True
DEBUG_NANS = False
VERBOSE = False

In [2]:
# Standard library imports
import os

os.environ['JAX_ENABLE_X64'] = str(USE_X64).lower()

import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

from abc import ABC, abstractmethod
from copy import deepcopy

In [3]:
# Third party
import jax

jax.config.update("jax_disable_jit", not USE_JIT)
jax.config.update("jax_debug_nans", DEBUG_NANS)

In [4]:
# Third party
from jax import jit, vmap
from jax import numpy as jnp
from jax import lax
from jax.lax import cond
import jax.tree_util as jtu
from equinox import filter_jit

import numpy as np
import pandas as pd

In [5]:
# Local
from kernax import SEKernel, AbstractKernel, DiagKernel, ExpKernel, BatchKernel
from MagmaClustPy.utils import preprocess_db
from MagmaClustPy.linalg import map_to_full_matrix_batch, map_to_full_array_batch, compute_mapping, lexicographic_sort
from MagmaClustPy.hyperpost import hyperpost
from MagmaClustPy.hp_optimisation import optimise_mean_kernel, optimise_task_kernel
from MagmaClustPy.prediction import predict
from MagmaClustPy.custom_kernels import RBFKernel, SEMagmaKernel

INFO:2025-12-03 15:50:02,330:jax._src.xla_bridge:752: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: dlopen(libtpu.so, 0x0001): tried: 'libtpu.so' (no such file), '/System/Volumes/Preboot/Cryptexes/OSlibtpu.so' (no such file), '/opt/miniconda3/envs/MagmaClustPy/bin/../lib/libtpu.so' (no such file), '/usr/lib/libtpu.so' (no such file, not in dyld cache), 'libtpu.so' (no such file)
2025-12-03 15:50:02,330 - INFO - Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: dlopen(libtpu.so, 0x0001): tried: 'libtpu.so' (no such file), '/System/Volumes/Preboot/Cryptexes/OSlibtpu.so' (no such file), '/opt/miniconda3/envs/MagmaClustPy/bin/../lib/libtpu.so' (no such file), '/usr/lib/libtpu.so' (no such file, not in dyld cache), 'libtpu.so' (no such file)


In [6]:
# Config
key = jax.random.PRNGKey(0)
test_db_size = "medium"

---
## Current implementation

In [7]:
# No current implementation

---
## Custom implementation(s)

### Utils

In [8]:
def check_db(db: pd.DataFrame):
	"""
	Makes preliminary checks on a database used for Magma(Clust) to prevent errors during runtime, providing more explicit error messages.
	:param db: the database to check
	"""
	pass

### Prior means

A prior mean is a function mapping inputs points (grid) to a prior value. The most common one is the zero prior mean. Other ones include linear, etc.

In [9]:
class BasePriorMean(ABC):
	@abstractmethod
	def __call__(self, grid):
		raise NotImplementedError

In [10]:
class ZeroMean(BasePriorMean):
	@filter_jit
	def __call__(self, grid):
		return jnp.zeros_like(grid[:, 0])

### Initializers

An initializer is a tool used at the start of the algorithm to find first guesses for kernel hyper-parameters values efficiently.

In [11]:
class BaseKernelInitializer(ABC):
	"""
	Kernel initializers are responsible for initializing kernel hyperparameters.

	They can adopt multiple strategise, using both data and likelihoods.
	"""

	def __init__(self):
		"""
		In subclasses, arguments used throughout the initialization should be passed as parameters to this constructor.
		This way, the class stores the info it needs, regardless of how/when it is called in the framework.
		"""
		pass

	@abstractmethod
	def init_kernel(self, kernel):
		raise NotImplementedError

In [12]:
class EmpiricalSEKernelInitializer(BaseKernelInitializer):
	@filter_jit
	def __init__(self, all_inputs, padded_outputs):
		super().__init__()

		diff = all_inputs[:, None, :] - all_inputs[None, :, :]
		dist_matrix = jnp.sqrt(jnp.sum(diff ** 2, axis=-1))
		upper_tri_dists = dist_matrix[jnp.triu_indices(len(all_inputs), k=1)]

		self.length_scale = jnp.median(upper_tri_dists)
		self.variance = jnp.nanvar(padded_outputs)

	def init_kernel(self, kernel):
		"""
		Initialize a `Contant * SEKernel` by setting Constant.value to the empirical variance of outputs and SEKernal.lengthscale to the median of distances between inputs

		:param kernel: A ProductKernel containing a ConstantKernel (for variance) as its left_kernel and an IsotropicKernel (for length_scale) as its right_kernel
		:return: the new SEKernel (it is also modified in memory)
		"""
		kernel.left_kernel.value = jnp.full(kernel.left_kernel.value.shape, self.variance)
		kernel.right_kernel.length_scale = jnp.full(kernel.right_kernel.length_scale.shape, self.length_scale)

		return kernel

### Likelihoods

The likelihood is a tool used to transform the *posterior distribution* of the model into a *__predictive__ posterior distribution*.

It incorporate the noise learnt during training/given by the user into the prediction.

In [13]:
class BaseLikelihood:
	pass

### Models

In [14]:
class BaseModel(ABC):
	@abstractmethod
	def load_train_data(self, db: pd.DataFrame):
		"""
		Loads training data from a database, populating train attributes.
		The training data is used to fit the model, aka learn the hyperparameters, mean process, etc.


		:param db: pandas DataFrame with columns "Task_ID", "Input", "Input_ID", "Output", "Output_ID"
		:return:
		"""
		raise NotImplementedError

	@abstractmethod
	def load_pred_data(self, db):
		"""
		Loads pred data from a database, populating pred attributes.
		The pred data is used to make predictions after fitting the model. The model is conditioned on the training data *and* the pred data.

		:param db:
		:return:
		"""
		raise NotImplementedError

	@abstractmethod
	def load_test_data(self, db: pd.DataFrame):
		"""
		Loads test data from a database, populating test attributes.
		The test data is used to compare predictions from the model (after fitting and conditioning on pred data) against ground truth values.
		Each ID in test data must coincide with an ID in pred data. Test data contains the points where outputs are known but we want to hide them from the model during fitting and prediction.

		:param db: pandas DataFrame with columns "Task_ID", "Input", "Input_ID", "Output", "Output_ID"
		:return:
		"""
		raise NotImplementedError

	@abstractmethod
	def fit(self):
		"""
		Fits the model to the training data.

		:return:
		"""
		raise NotImplementedError

	@abstractmethod
	def predict(self, X_test):
		"""
		Makes predictions on test data.

		:param X_test:
		:return:
		"""
		raise NotImplementedError

	@abstractmethod
	def plot_mean_process(self):
		"""
		Plots mean process.

		:return:
		"""
		raise NotImplementedError

	@abstractmethod
	def plot_predictions(self):
		"""
		Plots predictions.

		:return:
		"""
		raise NotImplementedError

In [15]:
class MagmaClust(BaseModel):
	def __init__(self,
	             k: int,
	             likelihood: BaseLikelihood,
	             prior_mean: BasePriorMean,
	             mean_kernel: AbstractKernel,
	             task_kernel_train: AbstractKernel,
	             task_kernel_pred: AbstractKernel,
	             shared_hp: bool):
		self.k = k
		self.likelihood = likelihood
		self.prior_mean = prior_mean
		self.mean_kernel = mean_kernel
		self.task_kernel_train = task_kernel_train
		self.task_kernel_pred = task_kernel_pred
		self.shared_hp = shared_hp

		# Attributes that will be instantiated later
		self.padded_inputs_train = None
		self.padded_outputs_train = None
		self.mappings_train = None
		self.all_inputs_train = None
		self.shared_inputs_train = None

		self.padded_inputs_pred = None
		self.padded_outputs_pred = None
		self.mappings_pred = None
		self.all_inputs_pred = None

		self.padded_inputs_test = None
		self.padded_outputs_test = None
		self.mappings_test = None
		self.all_inputs_test = None

		self.post_mean = None
		self.post_cov = None

	def load_train_data(self, db: pd.DataFrame, skip_check=False):
		if not skip_check:
			check_db(db)
		self.padded_inputs_train, self.padded_outputs_train, self.mappings_train, self.all_inputs_train = preprocess_db(
			db)
		self.shared_inputs_train = jnp.all(self.padded_inputs_train[0] == self.all_inputs_train).item()

		# Batch kernels, if they are not already batched
		if not isinstance(task_kernel_train, BatchKernel):
			if self.shared_hp:
				self.task_kernel_train = BatchKernel(self.task_kernel_train,
				                          batch_size=self.padded_inputs_train.shape[0], batch_in_axes=None, batch_over_inputs=True)
			else:
				self.task_kernel_train = BatchKernel(self.task_kernel_train,
				                          batch_size=self.padded_inputs_train.shape[0], batch_in_axes=0, batch_over_inputs=True)

	def load_pred_data(self, db: pd.DataFrame, skip_check=True):
		if not skip_check:
			check_db(db)
		self.padded_inputs_pred, self.padded_outputs_pred, self.mappings_pred, self.all_inputs_pred = preprocess_db(db)

		if not isinstance(task_kernel_pred, BatchKernel):
			if self.shared_hp:
				self.task_kernel_pred = BatchKernel(self.task_kernel_pred,
				                          batch_size=self.padded_inputs_pred.shape[0], batch_in_axes=None, batch_over_inputs=True)
			else:
				self.task_kernel_pred = BatchKernel(self.task_kernel_pred,
				                          batch_size=self.padded_inputs_pred.shape[0], batch_in_axes=0, batch_over_inputs=True)

	def load_test_data(self, db: pd.DataFrame, skip_check=True):
		if not skip_check:
			check_db(db)
		self.padded_inputs_test, self.padded_outputs_test, self.mappings_test, all_inputs_test = preprocess_db(db)

	def fit(self, max_iter: int = 25, converg_threshold: float = 1e-3, jitter: jnp.ndarray = jnp.array(1e-4)):
		# Monitoring variables
		prev_mean_llh = jnp.inf
		prev_task_llh = jnp.inf
		conv_ratio = jnp.inf

		for i in range(max_iter):
			logging.info(
				f"Iteration {i:4}\tLlhs: {prev_mean_llh:12.4f}, {prev_task_llh:12.4f}\tConv. Ratio: {conv_ratio:.5f}\t\n\tMean kernel: {self.mean_kernel}\n\tTask kernel: {self.task_kernel_train}")
			# e-step: compute hyper-posterior
			prior_mean_on_grid = self.prior_mean(self.all_inputs_train)
			self.post_mean, self.post_cov = hyperpost(self.padded_inputs_train, self.padded_outputs_train, self.mappings_train,
			                                self.all_inputs_train,
			                                prior_mean_on_grid, self.mean_kernel, self.task_kernel_train,
			                                shared_input=self.shared_inputs_train, shared_hp=self.shared_hp)

			# m-step: update hyperparameters
			self.mean_kernel, mean_llh = optimise_mean_kernel(self.mean_kernel, self.all_inputs_train, prior_mean_on_grid,
			                                             self.post_mean, self.post_cov, jitter=jitter)
			self.task_kernel_train, task_llh = optimise_task_kernel(self.task_kernel_train, self.padded_inputs_train, self.padded_outputs_train,
			                                             self.mappings_train, self.post_mean, self.post_cov, jitter=jitter)

			# Check for NaN values and stop early
			if jnp.isnan(mean_llh) or jnp.isnan(task_llh):
				logging.error(f"NaN detected at iteration {i}. Stopping training.")
				break

			# Check convergence
			if i > 0:
				conv_ratio = jnp.abs((prev_mean_llh + prev_task_llh) - (mean_llh + task_llh)) / jnp.abs(
					prev_mean_llh + prev_task_llh)
			if conv_ratio < converg_threshold:
				logging.info(
					f"Convergence reached after {i + 1} iterations.\tLlhs: {mean_llh:12.4f}, {task_llh:12.4f}\n\tMean kernel: {self.mean_kernel}\n\tTask kernel: {self.task_kernel_train}")
				break

			if i == max_iter - 1:
				logging.warning(
					f"Maximum number of iterations reached.\nLast modif: {jnp.abs(prev_mean_llh - mean_llh).item()} & {jnp.abs(prev_task_llh - task_llh).item()}")

			prev_mean_llh = mean_llh
			prev_task_llh = task_llh

	def optimise_pred_kernels(self, jitter: jnp.ndarray = jnp.array(1e-4)):
		# Optimise the task kernel for prediction
		self.task_kernel_pred, _ = optimise_task_kernel(self.task_kernel_pred, self.padded_inputs_pred, self.padded_outputs_pred,
		                                                self.mappings_pred, self.post_mean, self.post_cov, jitter=jitter)

	def predict(self, grid: np.ndarray, skip_retrain: bool=False) -> np.ndarray:
		if not self.shared_hp and not skip_retrain:
			self.optimise_pred_kernels()

		# Merge grid and all_inputs and compute new mappings
		full_grid = lexicographic_sort(jnp.unique(jnp.concatenate([self.all_inputs_train, self.all_inputs_pred, grid]), axis=0))
		# Compute new mappings
		mappings_train_on_grid = vmap(compute_mapping, in_axes=(None, 0))(full_grid, self.padded_inputs_train)
		mappings_pred_on_grid = vmap(compute_mapping, in_axes=(None, 0))(full_grid, self.padded_inputs_pred)

		# Compute the hyper-posterior on the grid
		post_mean_grid, post_cov_grid = hyperpost(inputs=self.padded_inputs_train,
		                                          outputs=self.padded_outputs_train,
		                                          mappings=mappings_train_on_grid,
		                                          all_inputs=full_grid,
		                                          prior_mean=jnp.array(0.),
		                                          mean_kernel=self.mean_kernel,
		                                          task_kernel=self.task_kernel_pred,
		                                          shared_input=False,  # As we use a grid
		                                          shared_hp=self.shared_hp)

		# Compute predictions
		return predict(post_mean_grid, post_cov_grid, self.padded_outputs_pred, mappings_pred_on_grid, full_grid, self.task_kernel_pred)

	def plot_predictions(self):
		pass

	def plot_mean_process(self):
		pass


In [16]:
class Magma(MagmaClust):
	"""
	Magma model class. It is just a MagmaClust with K=1.
	"""

	def __init__(self, **kwargs):
		super().__init__(k=1, **kwargs)

---
## Model use

### Config

In [17]:
# CONFIG
dataset = "medium"
shared_input = True
shared_hp = True
max_iter = 25
converg_threshold = 1e-3
grid_size = 100
jitter = jnp.array(1e-4)

### Data

In [18]:
db = pd.read_csv(f"../datasets/K=1/{dataset}_{"shared_input" if shared_input else "distinct_input"}_{"shared_hp" if shared_hp else "distinct_hp"}.csv")

db.head()

,Task_ID,Input,Output,se_variance,se_lengthscale,noise,Input_ID,Output_ID
0,1,-99.0,-462.168670,1.09,0.8,0.08,x,a
1,1,-98.0,-465.005425,1.09,0.8,0.08,x,a
2,1,-97.0,-449.896776,1.09,0.8,0.08,x,a
3,1,-96.5,-440.884203,1.09,0.8,0.08,x,a
4,1,-96.0,-437.432342,1.09,0.8,0.08,x,a


In [19]:
# First 90% of IDs are for training, last 10% for testing
train_ids = db["Task_ID"].unique()[:int(0.9 * db["Task_ID"].nunique())]
pred_ids = db["Task_ID"].unique()[int(0.9 * db["Task_ID"].nunique()):]

db_train = db[db["Task_ID"].isin(train_ids)]
db_pred = db[db["Task_ID"].isin(pred_ids)]

# First 70% of inputs of each pred_id is for pred, last 30% for test
db_test = db_pred.groupby("Task_ID", group_keys=False).apply(
    lambda x: x.iloc[int(0.7 * len(x)):]
)
db_pred = db_pred.groupby("Task_ID", group_keys=False).apply(
    lambda x: x.iloc[:int(0.7 * len(x))]
)

/var/folders/zr/lylpr6j91ls4k6rkcfn7dh680000gn/T/ipykernel_47122/2553002755.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  db_test = db_pred.groupby("Task_ID", group_keys=False).apply(
/var/folders/zr/lylpr6j91ls4k6rkcfn7dh680000gn/T/ipykernel_47122/2553002755.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  db_pred = db_pred.groupby("Task_ID", group_keys=False).apply(


### Initialisation

In [20]:
# TODO: use initializer
# TODO: noise kernel inside likelihood
mean_kernel = SEMagmaKernel(length_scale=jnp.array(0.9), variance=jnp.array(1.5))
task_kernel_train = SEMagmaKernel(length_scale=jnp.array(.3), variance=jnp.array(1.)) + DiagKernel(ExpKernel(jnp.array(2.5)))
task_kernel_pred = deepcopy(task_kernel_train)

### Here we go

In [21]:
model = Magma(likelihood=BaseLikelihood(), prior_mean=ZeroMean(), mean_kernel=mean_kernel, task_kernel_train=task_kernel_train, task_kernel_pred=task_kernel_pred, shared_hp=shared_hp)

In [22]:
model.load_train_data(db_train)

In [23]:
model.load_pred_data(db_pred)

In [24]:
model.load_test_data(db_test)

In [25]:
model.fit()

2025-12-03 15:50:05,288 - INFO - Iteration    0	Llhs:          inf,          inf	Conv. Ratio: inf	
	Mean kernel: SEMagmaKernel(length_scale=0.90, variance=1.50)
	Task kernel: SEMagmaKernel(length_scale=0.30, variance=1.00) + Diag(Exp(2.50))
2025-12-03 15:50:20,206 - INFO - Iteration    1	Llhs:     639.2806,   62745.9256	Conv. Ratio: inf	
	Mean kernel: SEMagmaKernel(length_scale=1.60, variance=10.45)
	Task kernel: SEMagmaKernel(length_scale=10.06, variance=2.87) + Diag(Exp(1.73))
2025-12-03 15:50:41,927 - INFO - Iteration    2	Llhs:     673.2206,   41920.0645	Conv. Ratio: 0.32802	
	Mean kernel: SEMagmaKernel(length_scale=1.50, variance=10.49)
	Task kernel: SEMagmaKernel(length_scale=0.97, variance=1.06) + Diag(Exp(-1.30))
2025-12-03 15:50:46,246 - INFO - Iteration    3	Llhs:     731.7932,   37920.4180	Conv. Ratio: 0.09253	
	Mean kernel: SEMagmaKernel(length_scale=1.27, variance=10.49)
	Task kernel: SEMagmaKernel(length_scale=0.85, variance=1.07) + Diag(Exp(-2.21))
2025-12-03 15:50:51,12

In [26]:
grid = jnp.linspace(jnp.min(model.all_inputs_train - 5., axis=0), jnp.max(model.all_inputs_train + 5., axis=0), grid_size)

In [27]:
model.predict(grid)

(Array([[ -29.71105016, -241.49689862, -449.556392  , ...,  426.13946444,
          130.42697124,    9.95608072],
        [ -29.78948599, -241.9194249 , -449.67764714, ...,  426.13946444,
          130.42697124,    9.95608072],
        [ -29.71406472, -241.50506025, -449.36140921, ...,  426.13946444,
          130.42697124,    9.95608072],
        ...,
        [ -29.72627679, -241.58567346, -449.63360806, ...,  426.13946444,
          130.42697124,    9.95608072],
        [ -29.71002841, -241.49403467, -449.33807623, ...,  426.13946444,
          130.42697124,    9.95608072],
        [ -29.72802262, -241.60088764, -449.81543451, ...,  426.13946444,
          130.42697124,    9.95608072]], dtype=float64),
 Array([[[ 3.48538462e+04,  1.58915186e+04,  7.06759979e+02, ...,
          -1.91407678e-21, -2.21125228e-21, -3.38985096e-22],
         [ 1.58915186e+04,  2.21130178e+04,  2.05814283e+03, ...,
          -1.13264566e-20, -1.30569605e-20, -1.93798500e-21],
         [ 7.06759979e+02,  2.

---
## Conclusion

---